# Tipping Point 1D Plotting, one product

Plug and play version. Put the two DPM exports in a folder, set that folder in Section 2, then
Run All. Nothing else needs editing.

## Requirements

**Requirement 1. File names.** Both exports must be named to this convention, because the pipeline
uses the names to work out which file is which and what the product is called:

```
DetailedModelResults_<Product>_TippingPoint.xlsx     the swept results
DetailedModelResults_<Product>_MasterModels.xlsx     the B01 master models
```

For example `DetailedModelResults_Product_X_TippingPoint.xlsx` and
`DetailedModelResults_Product_X_MasterModels.xlsx`. `<Product>` is free text, so `Grizzly_XL`,
`Velo_Max`, `Glo` and `Vuse` all work, and it must be spelled the same way in both file names.

**Requirement 2. Model Group.** The `Model Group` column of both exports must read
`<Product>_<your initials>`, for example `Product_X_YO`. The pipeline splits on the final
underscore: everything before it is the product, which is checked against the file names, and the
last token is taken as the analyst's initials and printed on the report.

Anything that breaks either requirement stops the run immediately with a message naming the file
and what was expected, rather than failing several steps later with something obscure.

## What the pipeline does

It builds the four one dimensional tipping point figures used in the DPM modelling report. Each
figure plots the difference in survivors against the swept transition probability, faceted by RERR
and gateway effect, with the mean curve and the two 95% posterior interval curves. Where a curve
crosses zero is its tipping point, interpolated between the two swept probabilities that bracket
zero and printed on the figure. The diamond marks where the mean curve reaches the master model
survivor value.

| Figure | Sub question | Batch tag | Transition |
|---|---|---|---|
| 3 | 5a | `2 AI` | Additional initiation |
| 4 | 6a | `2 DS` | Diversion from smoking |
| 5 | 14b | `2 S` | Switching |
| 6 | 15a | `2 DQ` | Diversion from quitting |

## Sections

| Section | What it does | Why it is there |
|---|---|---|
| 1 | Imports and constants | One place for every setting that is not a folder path |
| 2 | Configuration | The only cell that normally changes between runs |
| 3 | Input resolution, Requirements 1 and 2 | Finds the two files, derives the product, fails early and clearly |
| 4 | Reading and preparing the exports | Handles either separator spelling, filters to the reporting rows, reshapes long |
| 5 | Tipping point calculation | The zero crossing and the master model crossing, one function for both |
| 6 | The figure builder | One figure per sub question, colour or greyscale |
| 7 | Saving the figures | Writes `color/` and `gray/`, svg and png |
| 8 | Key takeaways | Generated from the numbers, so the text cannot drift from the figure |
| 9 | The PDF report | Metadata page, then one page per figure with its takeaways |
| 10 | QC table | Every tipping point in one flat file |
| 11 | Orchestrator and run | Ties it together, this is the cell that produces everything |
| 12 | Export bundle | Freezes the run into a dated, versioned folder |
| 13 | Validation | Asserts the run is internally consistent and every promised file exists |
| 14 | Troubleshooting | Every error message the pipeline can raise, and what to do about it |

## Input files

| # | File | Sheet | Contents |
|---|---|---|---|
| 1 | `DetailedModelResults_<Product>_TippingPoint.xlsx` | `Detailed Results` | Node `DIFF_ALL`, age `68 - 72`, the four sweep batches for G10 and G25, both RERRs, both mortality models. Model names end in the swept probability |
| 2 | `DetailedModelResults_<Product>_MasterModels.xlsx` | `Detailed Results` | The B01 two age master models. Supplies the survivor value the diamond sits on |

Both have a title row above the real header, and in some exports the upper half of the 95% posterior
interval pair has no header at all. Both cases are handled.

## Output files

Everything is written under the folder set in Section 2.

| # | Path | Contents |
|---|---|---|
| 1 | `color/<Product>_AddInit_Mean95PIa.svg` and `.png` | Figure 3, additional initiation |
| 2 | `color/<Product>_DivSmo_Mean95PIa.svg` and `.png` | Figure 4, diversion from smoking |
| 3 | `color/<Product>_Switch_Mean95PIa.svg` and `.png` | Figure 5, switching |
| 4 | `color/<Product>_DivQuit_Mean95PIa.svg` and `.png` | Figure 6, diversion from quitting |
| 5 | `gray/` | The same four figures in greyscale, matching the all black styling of the R original |
| 6 | `<Product>_TippingPoints.csv` | QC table, every tipping point behind every figure |
| 7 | `Tipping Point_1D_Plotting_&_keyTakeways.pdf` | Metadata page, then one page per figure with key takeaways |
| 8 | `Exports_v<version>_<DDMonYYYY>/` | A frozen copy of all of the above, plus `MANIFEST.txt` |

## Provenance

Translated from `Tipping_Point_1D_Plotting_oneProduct_08242026.R`. `readxl` becomes
`pandas.read_excel`, `dplyr` and `tidyr` become pandas, `ggplot2` and `ggrepel` become a matplotlib
panel grid with deterministic label placement, `stats::lm` becomes a closed form two point solve plus
`numpy.polyfit`, and `grDevices::svg` becomes `Figure.savefig`. Every `NOTE:` fix flagged in the R
file is carried over; they change the numbers, so they are called out at the step they affect.

## Section 1. Imports and constants

**Rationale.** Everything that is not a folder path lives here, so Section 2 stays short enough to
be the only thing anyone edits. `matplotlib` is put into the `Agg` backend so the notebook writes
identical files whether or not a display is attached.

The two dictionaries worth knowing about:

* `SUBQ_TAGS` maps the tag inside a model name to the sub question. The names are matched after
  separators are normalised, so `_14b_` finds both `..._2 S_14b_0.00` and `... 2 S 14b 0.00`.
* `FIGSPEC` holds the per figure tuning from the four `plot1()` calls at the end of the R script:
  the file name tag, the x axis breaks, and the two label placement knobs.

In [ ]:
# %pip install pandas numpy openpyxl matplotlib

from __future__ import annotations

import getpass
import platform
import re
import shutil
import textwrap
import warnings
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib
matplotlib.use("Agg")            # write files without needing a display
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib.lines import Line2D

# ------------------------------------------------------------------- inputs
SHEET = "Detailed Results"        # the only sheet either export carries

# Requirement 1. The two file name patterns, and how the product is read out of them.
FILE_PATTERN = "DetailedModelResults_{product}_{role}.xlsx"
ROLE_SUFFIX = {"sweeps": "TippingPoint", "master": "MasterModels"}
FILE_REGEX = re.compile(r"^DetailedModelResults_(?P<product>.+)_(?P<role>TippingPoint|MasterModels)\.xlsx$")

# Columns whose VALUES keep their spacing. An age range reads as a range, so underscoring
# '68 - 72' to '68_-_72' would make it unreadable on a figure and in the QC table.
VALUE_EXEMPT_COLUMNS = ("Age_Range",)

# ------------------------------------------------------------- figure content
# Sub question tags, in the order the R script tested them
SUBQ_TAGS = [("_5a_", "5a"), ("_6a_", "6a"), ("_15a_", "15a"), ("_14b_", "14b")]

XLABELS = {"5a": "Additional initiation (%)", "6a": "Diversion from smoking (%)",
           "14b": "Switching (%)", "15a": "Diversion from quitting (%)"}
TRANSITIONS = {"5a": "Additional initiation", "6a": "Diversion from smoking",
               "14b": "Switching", "15a": "Diversion from quitting"}
STAT_LABELS = {"Mean": "Mean", "95%_PI": "95% PI lower", "95%_PI2": "95% PI upper"}

# Per figure settings from the four plot1() calls in the R script:
# (file name tag, x axis breaks or None for the swept probabilities, label push direction, force)
FIGSPEC = {
    "5a":  ("AddInit", [0, 5, 10, 15, 20], 2.5, 2.0),
    "6a":  ("DivSmo", None, -1.0, 1.5),
    "14b": ("Switch", None, -0.5, 1.5),
    "15a": ("DivQuit", None, 1.0, 1.5),
}
FIGURE_ORDER = ["5a", "6a", "14b", "15a"]
FIGURE_NUMBERS = {"5a": 3, "6a": 4, "14b": 5, "15a": 6}

# Priority when duplicate tipping point labels are collapsed: the mean wins
PRIORITY = {"Mean": 1, "95%_PI": 2, "95%_PI2": 3}

# Colour and greyscale palettes. The grey one reproduces the all black styling of the R original.
PALETTES = {
    "color": {"mean": "#1F4E79", "pi": "#C00000", "mm": "#E26B0A",
              "zero": "#808080", "strip": "#DCE6F1", "grid": "#D9D9D9"},
    "gray":  {"mean": "#000000", "pi": "#6E6E6E", "mm": "#000000",
              "zero": "#B0B0B0", "strip": "#E8E8E8", "grid": "#DCDCDC"},
}

# ------------------------------------------------------------------- outputs
COLOR_DIR_NAME = "color"
GRAY_DIR_NAME = "gray"
EXPORT_DIR_STEM = "Exports"
PDF_NAME = "Tipping Point_1D_Plotting_&_keyTakeways.pdf"

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

print(f"pandas {pd.__version__} | numpy {np.__version__} | matplotlib {matplotlib.__version__}")

## Section 2. Configuration

**Rationale.** One folder, a handful of filters, nothing else. The product name is deliberately not
configurable: it comes from the file names and is checked against `Model Group`, so it cannot be set
to something the data does not support.

Set `DATA_DIR` and run every cell. `OUT_DIR = None` writes the results next to the inputs, which is
usually what you want.

In [ ]:
# The folder holding the two DPM exports. Use a raw string (r"...") on Windows so the backslashes
# are not read as escape characters.
DATA_DIR = r"C:\Users\YemiOdeyemi\Downloads\Grizzly\Tipping_Point_1D"
OUT_DIR = None                   # None writes the outputs into DATA_DIR

# Rows that define a valid record. Written the way they read in the export; separators are
# normalised before comparison, so "DIFF_ALL" also matches an export that writes "DIFF ALL".
NODE_KEEP = "DIFF_ALL"
AGE_RANGE_KEEP = "68 - 72"
ERR_KEEP = [0.05, 0.10]          # compared numerically, so 0.1 and 0.10 both match
MORTALITY_MODEL_PLOT = "Male"    # the report figures use the male model, set "Female" to switch

# Figure page size in inches, matching the report figure boxes
FIG_WIDTH, FIG_HEIGHT = 7.5, 10.0
FIG_DPI = 200                    # raster resolution for the .png copies

# Report provenance, printed on the first page of the PDF
AUTHOR_NAME = None               # None uses the initials from Model Group, then the account name
PIPELINE_VERSION = "3.0"
VERSION_TYPE = "Final"           # for example Draft, QC or Final
ANALYSIS_OBJECTIVE = (
    "Identify the transition probability at which the modelled survivor difference crosses zero "
    "(the tipping point) for additional initiation, diversion from smoking, switching and "
    "diversion from quitting, and locate the master model result on each swept curve.")

## Section 3. Input resolution, Requirements 1 and 2

**Rationale.** Both requirements exist so the pipeline can be plug and play, and this section is
where they earn their keep. Requirement 1 means the two files identify their own roles and carry the
product name, so nothing has to be typed twice or kept in sync by hand. Requirement 2 means the
product can be confirmed against the data itself rather than trusted from a file name, and it
supplies the analyst initials for the report.

Order of checks, each failing with a message that names the file and the expected form:

1. Find exactly one file per role in the folder, matching the Requirement 1 pattern.
2. Confirm both file names carry the same product.
3. Read `Model Group` from each export and split off the final token as the initials.
4. Confirm the product from `Model Group` matches the product from the file names, ignoring
   spacing, underscores, hyphens and case.
5. Confirm each file actually holds what its name claims: a sweep file's model names end in the
   swept probability, a master model file's end in a cohort tag such as `B01`. This is the check
   that catches two workbooks saved under each other's names, which otherwise surfaces much later
   as `rows have an unparseable swept probability`.

`normalise_separators` is what lets any of this work on either spelling of the exports. Some DPM
downloads write `Master Model G10 Product X 2 S 14b 0.00` and `JAGS Male 2000`, others write the
same values with underscores. Runs of whitespace are converted to single underscores so both read
identically; `Age_Range` is exempt so `68 - 72` stays legible.

In [ ]:
WHITESPACE = re.compile(r"\s+")


def normalise_separators(value):
    """'Master Model G10 Product X 2 S 14b 0.00' -> 'Master_Model_G10_Product_X_2_S_14b_0.00'.

    Trims, then collapses every run of whitespace to a single underscore. Non text values come
    back untouched, so numbers and timestamps survive. Already underscored input is unchanged,
    which is what makes the function safe to apply twice.
    """
    if not isinstance(value, str):
        return value
    return WHITESPACE.sub("_", value.strip())


def same_label(a, b) -> bool:
    """Compare two labels ignoring spacing, underscores, hyphens and case."""
    key = lambda s: re.sub(r"[\s_\-]+", "", str(s)).casefold()
    return key(a) == key(b)


def safe_name(value) -> str:
    """Make a label safe in a file name: separators to underscore, then drop illegal characters."""
    return re.sub(r'[<>:"/\\|?*]+', "_", normalise_separators(value)) or "Product"


def read_dpm(path, sheet: str = SHEET) -> pd.DataFrame:
    """Read one DPM export and normalise it enough to be worked with.

    Three things happen, all of them needed before any column can be trusted:
      * the header is taken from row 2, since the DPM writes a title in row 1;
      * the upper half of the 95% posterior interval pair is named by position, because some
        exports leave it with no header at all;
      * column names and text values have their separators normalised, Age_Range excepted.
    Everything is read as objects on purpose: some exports repeat the header row partway down the
    sheet, and reading as text keeps those rows from raising a type error before they are filtered.
    """
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Input file not found:\n  {path}")

    raw = pd.read_excel(path, sheet_name=sheet, header=1, dtype=object)
    cols = [str(c).strip() for c in raw.columns]
    norm = [normalise_separators(c) for c in cols]
    if "95%_PI" not in norm or norm.index("95%_PI") + 1 >= len(cols):
        raise ValueError(f"Could not locate the '95% PI' column pair in {path.name}. "
                         f"Check that row 1 is the title and row 2 holds the column names.")
    norm[norm.index("95%_PI") + 1] = "95%_PI2"
    raw.columns = norm

    exempt = {normalise_separators(c) for c in VALUE_EXEMPT_COLUMNS}
    for col in raw.columns:
        if raw[col].dtype != object:
            continue
        if col in exempt:
            raw[col] = raw[col].map(lambda v: v.strip() if isinstance(v, str) else v)
        else:
            raw[col] = raw[col].map(normalise_separators)

    for need in ("Model_Group", "Model_Name", "Mortality_Model", "ERR", "Node", "Age_Range", "Mean"):
        if need not in raw.columns:
            raise KeyError(f"{path.name} is missing the required column '{need.replace('_', ' ')}'.")
    return raw


def looks_like_sweeps(df: pd.DataFrame) -> float:
    """Share of model names ending in a number, which is what separates a sweep from a master."""
    tail = df["Model_Name"].astype(str).str.rsplit("_", n=1).str[-1]
    return float(pd.to_numeric(tail, errors="coerce").notna().mean())


def split_model_group(model_group) -> tuple[str, str]:
    """Requirement 2: 'Product_X_YO' -> ('Product_X', 'YO'), product and analyst initials."""
    value = normalise_separators(model_group)
    tokens = [t for t in value.split("_") if t]
    if len(tokens) < 2:
        raise ValueError(
            f"Model Group is {model_group!r}, which does not meet Requirement 2. It must read "
            "'<Product>_<your initials>', for example 'Product_X_YO'.")
    return "_".join(tokens[:-1]), tokens[-1]


def resolve_inputs(data_dir=None, out_dir=None) -> dict:
    """Find both exports, check Requirements 1 and 2, and return the paths and labels."""
    data_dir = Path(data_dir or DATA_DIR)
    if not data_dir.is_dir():
        raise NotADirectoryError(f"DATA_DIR does not exist:\n  {data_dir}")
    out_dir = Path(out_dir) if out_dir else data_dir

    # ---- Requirement 1: exactly one file per role, both naming the same product
    found = {}
    for path in sorted(data_dir.glob("DetailedModelResults_*.xlsx")):
        if path.name.startswith("~$"):          # an Excel lock file, not a real workbook
            continue
        m = FILE_REGEX.match(path.name)
        if m:
            role = "sweeps" if m["role"] == "TippingPoint" else "master"
            found.setdefault(role, []).append((path, m["product"]))

    for role, suffix in ROLE_SUFFIX.items():
        hits = found.get(role, [])
        if len(hits) != 1:
            wanted = FILE_PATTERN.format(product="<Product>", role=suffix)
            raise FileNotFoundError(
                f"Requirement 1 not met: expected exactly one '{wanted}' in {data_dir}, "
                f"found {len(hits)}"
                + (f" ({', '.join(p.name for p, _ in hits)})." if hits else "."))

    paths = {role: hits[0][0] for role, hits in found.items()}
    products = {role: hits[0][1] for role, hits in found.items()}
    if not same_label(products["sweeps"], products["master"]):
        raise ValueError(
            f"Requirement 1 not met: the two file names carry different products, "
            f"{products['sweeps']!r} and {products['master']!r}. They must match.")
    product = normalise_separators(products["sweeps"])

    # ---- Requirement 2: Model Group is '<Product>_<initials>' and agrees with the file names
    frames, initials = {}, {}
    for role, path in paths.items():
        frame = read_dpm(path)
        groups = [g for g in frame["Model_Group"].dropna().unique() if g != "Model_Group"]
        if len(groups) != 1:
            raise ValueError(f"{path.name} contains {len(groups)} model groups ({groups}). "
                             "This is the one product pipeline; split the export.")
        grp_product, grp_initials = split_model_group(groups[0])
        if not same_label(grp_product, product):
            raise ValueError(
                f"Requirement 2 not met: {path.name} names the product {product!r} but its "
                f"Model Group is {groups[0]!r}, giving {grp_product!r}. Fix whichever is wrong.")
        frames[role], initials[role] = frame, grp_initials

    if initials["sweeps"] != initials["master"]:
        warnings.warn(f"The two exports carry different initials in Model Group, "
                      f"{initials['sweeps']!r} and {initials['master']!r}. Using the sweep file's.")

    # ---- Contents must match the role the file name claims
    share = {role: looks_like_sweeps(frame) for role, frame in frames.items()}
    if share["sweeps"] < 0.9 or share["master"] > 0.5:
        raise ValueError(
            "The two workbooks do not hold what their names claim. A TippingPoint file's model "
            "names should end in the swept probability, a MasterModels file's in a cohort tag "
            f"such as B01.\n  {paths['sweeps'].name}: {share['sweeps']:.0%} of names end in a "
            f"number\n  {paths['master'].name}: {share['master']:.0%}\n"
            "The two files are most likely saved under each other's names.")

    print(f"Product : {product}      Analyst initials: {initials['sweeps']}")
    for role in ("sweeps", "master"):
        print(f"  {role:<7} {len(frames[role]):>4} rows   {paths[role].name}")
    print(f"  output folder: {out_dir}")
    return {"product": product, "initials": initials["sweeps"], "paths": paths,
            "frames": frames, "data_dir": data_dir, "out_dir": out_dir}

## Section 4. Reading and preparing the exports

**Rationale.** Both workbooks go through the same filters, which is what makes their join keys line
up later. Doing it in one shared function means a filter can never be applied to one file and
forgotten on the other, which was the failure mode the R script's `NOTE (join)` warned about.

Steps and why each one is there:

1. Drop repeated header rows, keep the reporting node and age range. Some exports repeat the header
   partway down the sheet, and the model runs other nodes and age ranges that are not reported.
2. Keep the two RERRs, compared numerically so `0.1` and `0.10` both match, then normalise RERR to a
   fixed two decimal string. Fixed formatting is what makes the tipping point and master model files
   join reliably.
3. Map the mortality model to `Male` or `Female` and keep the one being plotted. The female runs are
   exported for completeness and are not in the report figures.
4. Derive `Gateway` from `G10` or `G25` in the model name, and `GatewayDesc`, the printed caption.

The sweep file then gains `SubQ` from its batch tag, `xlabel` from that, `Prob` from the trailing
token of the model name, and `Key`, one sweep per gateway, mortality model, RERR and sub question.
Finally it is reshaped long, one row per statistic, which is `pivot_longer(c(Mean, 95% PI, 95% PI2))`.

Identifiers here are normalised; `GatewayDesc` and `xlabel` keep ordinary spacing because they are
prose printed on the figure rather than anything that gets matched.

In [ ]:
def _common_filters(df: pd.DataFrame, label: str, product: str) -> pd.DataFrame:
    """Filters and derived columns both exports share."""
    df = df[df["Model_Group"].astype(str) != "Model_Group"]              # repeated header rows
    df = df[df["Node"].astype(str).map(lambda v: same_label(v, NODE_KEEP))]
    df = df[df["Age_Range"].astype(str).map(lambda v: same_label(v, AGE_RANGE_KEEP))]

    err = pd.to_numeric(df["ERR"], errors="coerce")
    df = df[err.round(4).isin(np.round(ERR_KEEP, 4))].copy()
    df["ERR"] = pd.to_numeric(df["ERR"], errors="coerce").map(lambda v: f"{v:.2f}")

    df["Mortality_Model"] = df["Mortality_Model"].astype(str).map(
        {"JAGS_Male_2000": "Male", "JAGS_Female_2000": "Female"})
    df = df[df["Mortality_Model"] == MORTALITY_MODEL_PLOT].copy()

    if df.empty:
        raise ValueError(
            f"No rows left after filtering the {label} export. Check NODE_KEEP, AGE_RANGE_KEEP, "
            "ERR_KEEP and MORTALITY_MODEL_PLOT in Section 2 against the workbook.")

    name = df["Model_Name"].astype(str)
    df["Gateway"] = np.where(name.str.contains("G10"), "G10",
                             np.where(name.str.contains("G25"), "G25", None))
    if df["Gateway"].isna().any():
        raise ValueError(f"{int(df['Gateway'].isna().sum())} rows in the {label} export have "
                         "neither 'G10' nor 'G25' in the model name.")
    df["GatewayDesc"] = np.where(df["Gateway"] == "G10",
                                 "Gateway effect/return\nto smoking = 10%",
                                 "Gateway effect/return\nto smoking = 25%")
    df["Product"] = product
    df["plabel"] = product
    df["Strength"] = product        # these exports have no strength dimension
    return df


def prepare_sweeps(raw: pd.DataFrame, product: str) -> pd.DataFrame:
    """Filter, derive and reshape the tipping point export into one row per statistic."""
    df = _common_filters(raw, "tipping point", product)
    name = df["Model_Name"].astype(str)

    subq = pd.Series(pd.NA, index=df.index, dtype=object)
    for tag, lab in SUBQ_TAGS:
        subq = subq.where(subq.notna(), np.where(name.str.contains(tag, regex=False), lab, None))
    df["SubQ"] = subq
    df["xlabel"] = df["SubQ"].map(XLABELS)

    # The swept probability is the trailing token: '..._2_AI_5a_0.08' -> 0.08
    df["Prob"] = pd.to_numeric(name.str.rsplit("_", n=1).str[-1], errors="coerce")

    if df["SubQ"].isna().any():
        raise ValueError(f"{int(df['SubQ'].isna().sum())} rows have no sub question tag. "
                         "Expected '5a', '6a', '14b' or '15a' in the model name.")
    if df["Prob"].isna().any():
        raise ValueError(f"{int(df['Prob'].isna().sum())} rows have an unparseable swept "
                         "probability. Expected the model name to end in a number.")

    df["Key"] = (df["Gateway"] + ", " + df["Mortality_Model"].astype(str) + ", "
                 + df["ERR"] + ", " + df["SubQ"].astype(str))

    for c in ("Mean", "95%_PI", "95%_PI2"):
        df[c] = pd.to_numeric(df[c], errors="coerce")

    long = df.melt(
        id_vars=["Key", "Mortality_Model", "ERR", "Product", "Strength", "Gateway", "SubQ",
                 "Prob", "xlabel", "plabel", "GatewayDesc"],
        value_vars=["Mean", "95%_PI", "95%_PI2"],
        var_name="DataType", value_name="Result")
    long["DataTypeDesc"] = np.where(long["DataType"] == "Mean", "Mean", "95% posterior interval")
    long["Probability"] = 100 * long["Prob"]        # the sweeps are stored as fractions
    return long


def prepare_master(raw: pd.DataFrame, product: str) -> pd.DataFrame:
    """Filter the master model export down to the rows the diamond markers come from."""
    df = _common_filters(raw, "master model", product)
    df["Mean"] = pd.to_numeric(df["Mean"], errors="coerce")
    return df

## Section 5. Tipping point calculation

**Rationale.** One function serves both cases, because they are the same calculation with a
different target: zero for a tipping point, the master model survivor value for the diamond.

Method, matching `calcTPs()` and `calcTPs.MM()` step for step:

1. Sort the swept results ascending, ties keeping their original order.
   `numpy.argsort(kind="stable")` is exactly `rank(ties.method = "first")` then `order(rank)`. The R
   script's `NOTE (ties)` explains why: average ranks return half integers that never match the
   bracket index, which silently collapsed a bracketing pair to one row.
2. Find the pair bracketing the target. R's `findInterval` on an ascending vector returns the count
   of values at or below the target, which `numpy.searchsorted(..., side="right")` reproduces.
3. With a bracketing pair, solve the straight line through those two points for the crossing. A two
   point `lm()` is an exact line, so the closed form gives the same answer as the regression.
4. With no bracketing pair the curve never crosses inside the swept range, and the direction comes
   from the slope fitted through **all** the points. The R script's `NOTE (direction)` is the reason:
   assuming survivors always fall as the probability rises is backwards for switching and diversion
   from smoking, where a curve that is positive throughout crosses zero **below** the smallest swept
   probability.

Each interval bound is ranked on its own values, not on the mean, which is the third `NOTE:` fix.
The slope is returned alongside the label because Section 8 uses its sign to describe the direction
in words.

In [ ]:
def r_signif(x, digits: int = 3) -> float:
    """R's signif(): round to significant digits, not decimal places. round() is not a substitute."""
    x = float(x)
    if x == 0 or not np.isfinite(x):
        return x
    return float(round(x, -int(np.floor(np.log10(abs(x)))) + (digits - 1)))


def num_str(x) -> str:
    """Format a number the way R's paste0() would: 20.0 -> '20', 0.05 -> '0.05'."""
    return f"{float(x):g}"


def fit_slope(x, y) -> float:
    """Slope through every point, R's coef(lm(y ~ x))[2]."""
    x, y = np.asarray(x, float), np.asarray(y, float)
    ok = np.isfinite(x) & np.isfinite(y)
    if ok.sum() < 2 or np.ptp(x[ok]) == 0:
        return np.nan
    return float(np.polyfit(x[ok], y[ok], 1)[0])


def tipping_point(prob, values, target: float = 0.0, suffix: str = "%"):
    """Interpolate where a swept curve reaches `target`. Returns (label, slope).

    The label is a float when the curve crosses inside the swept range, and a string such as
    '>20%' or '<0%' when it does not.
    """
    prob, values = np.asarray(prob, float), np.asarray(values, float)
    if prob.size < 2:
        raise ValueError("A sweep needs at least two swept probabilities.")

    order = np.argsort(values, kind="stable")      # rank(ties='first') then order(rank)
    v, p = values[order], prob[order]
    slope_all = fit_slope(prob, values)

    n = int(np.searchsorted(v, target, side="right"))      # R's findInterval
    if 1 <= n < len(v):
        x1, x2, y1, y2 = p[n - 1], p[n], v[n - 1], v[n]
        if x1 != x2 and y1 != y2:
            slope = (y2 - y1) / (x2 - x1)                  # the two point lm(), solved exactly
            intercept = y1 - slope * x1
            return r_signif((target - intercept) / slope, 3), slope_all

    # No crossing inside the swept range. Direction from the fitted slope, not an assumption.
    if bool(v[0] > target) != bool(slope_all > 0):         # R's xor()
        return f">{num_str(prob.max())}{suffix}", slope_all
    return f"<{num_str(prob.min())}{suffix}", slope_all


def calc_tps(long: pd.DataFrame) -> pd.DataFrame:
    """Zero crossing tipping point for the mean and both interval bounds, one row each."""
    rows = []
    for key, g in long.groupby("Key", sort=False):
        wide = g.pivot_table(index="Probability", columns="DataType", values="Result",
                             aggfunc="mean").reset_index()
        meta = g.iloc[0]
        lo, hi = float(wide["Probability"].min()), float(wide["Probability"].max())

        for dt in ("Mean", "95%_PI", "95%_PI2"):
            if dt not in wide.columns:
                raise ValueError(f"Sweep '{key}' has no '{dt}' column after reshaping.")
            label, slope = tipping_point(wide["Probability"], wide[dt], 0.0, "%")
            # x_pos is where the label is drawn. Out of range labels have no numeric x, so they
            # are pinned to the end of the swept range they fall beyond, which is the R script's
            # NOTE about coercing Result to numeric only after the loop and losing them.
            x_pos = hi if (isinstance(label, str) and label.startswith(">")) else (
                    lo if isinstance(label, str) else float(label))
            rows.append({"Key": key, "Mortality_Model": meta["Mortality_Model"],
                         "SubQ": meta["SubQ"], "ERR": meta["ERR"], "Product": meta["Product"],
                         "plabel": meta["plabel"], "Strength": meta["Strength"],
                         "Gateway": meta["Gateway"], "GatewayDesc": meta["GatewayDesc"],
                         "DataType": dt, "Result": label, "x_pos": x_pos, "Slope": slope,
                         "SweepMin": lo, "SweepMax": hi})

    tps = pd.DataFrame(rows)
    tps["DataTypeDesc"] = np.where(tps["DataType"] == "Mean", "Mean", "95% posterior interval")
    return tps


# The keys the two exports are joined on, named explicitly. The R original called merge() with no
# `by`, so it joined on every column the two frames happened to share.
MM_JOIN = ["Mortality_Model", "ERR", "Product", "plabel", "Strength", "Gateway", "GatewayDesc"]


def calc_tps_mm(long: pd.DataFrame, mmdat: pd.DataFrame) -> pd.DataFrame:
    """Swept probability at which the mean curve reaches the master model survivor value."""
    left = long[long["DataType"] == "Mean"]
    right = mmdat[MM_JOIN + ["Mean"]].rename(columns={"Mean": "MM.survivor"})
    merged = left.merge(right, on=MM_JOIN, how="inner")
    if merged.empty:
        raise ValueError("The tipping point and master model exports did not join on any row. "
                         "Check that both cover the same RERRs, gateways and mortality model.")

    rows = []
    for key, g in merged.groupby("Key", sort=False):
        targets = g["MM.survivor"].unique()
        if len(targets) != 1:
            raise ValueError(f"Key '{key}' matched {len(targets)} master model means. The master "
                             "model export has a duplicate run for that gateway and RERR.")
        wide = g.pivot_table(index="Probability", values="Result", aggfunc="mean").reset_index()
        # No '%' suffix: this is a position on the sweep, not a tipping point. The R script's NOTE
        # about hard coded '>100' / '<0' applies here, the real sweep endpoints are used instead.
        label, slope = tipping_point(wide["Probability"], wide["Result"], float(targets[0]), "")
        meta = g.iloc[0]
        rows.append({"Key": key, "Mortality_Model": meta["Mortality_Model"], "ERR": meta["ERR"],
                     "Product": meta["Product"], "Strength": meta["Strength"],
                     "plabel": meta["plabel"], "Gateway": meta["Gateway"],
                     "GatewayDesc": meta["GatewayDesc"], "MM.prob": label,
                     "MM.survivor": float(targets[0]), "SubQ": meta["SubQ"],
                     "DataType": "Mean", "DataTypeDesc": "Mean", "Slope": slope})
    return pd.DataFrame(rows)

## Section 6. The figure builder

**Rationale.** The R script used `facet_grid(RERR + GatewayDesc ~ Strength)`, four rows and one
column, so this is a four panel column sharing both axes. Each panel carries the zero line, the mean
curve, the two interval curves, the master model diamond and the tipping point labels. The right
hand strip repeats the RERR and gateway caption, the top strip carries the product.

`ggrepel` has no maintained Python equivalent worth depending on, so label placement is
reimplemented: the same per statistic nudges as the R script, then a declutter pass, then a clamp
that keeps every label inside its panel. Being deterministic it needs no random seed, which is what
the R script's `seed = 42` was buying.

Two behaviours worth knowing:

* Duplicate labels are collapsed, preferring the mean. Where all three statistics fall outside the
  swept range they carry identical text at an identical x, and printing all three is an unreadable
  overprint that says nothing the single label does not.
* The label connector anchors to the zero line, but is pulled to the nearest visible y when the
  panel's range excludes zero. Matplotlib silently drops an entire annotation whose anchor sits
  outside the axes, so a `<0%` label would otherwise vanish with no warning at all.

In [ ]:
def tp_label(result) -> str:
    """'>20%' and '<0%' survive as written, a numeric tipping point becomes '10.2%'."""
    s = str(result)
    return s if s.startswith((">", "<")) else f"{float(result):.1f}%"


def _facet_order(dat: pd.DataFrame):
    """Panels run RERR ascending then gateway, matching facet_grid row order."""
    combos = dat[["ERR", "GatewayDesc"]].drop_duplicates()
    return list(combos.sort_values(["ERR", "GatewayDesc"]).itertuples(index=False, name=None))


def build_figure(dat: pd.DataFrame, mm_init: pd.DataFrame, tps: pd.DataFrame, subq: str,
                 palette: str = "color", figsize=None, ldir: float = 1.0, frc: float = 1.0,
                 xbreak=None, takeaways=None, title=None):
    """Build one tipping point figure. Returns a matplotlib Figure; the caller closes it."""
    if palette not in PALETTES:
        raise ValueError(f"palette must be one of {list(PALETTES)}, got {palette!r}.")
    pal = PALETTES[palette]
    figsize = figsize or (FIG_WIDTH, FIG_HEIGHT)

    d = dat[dat["SubQ"] == subq]
    if d.empty:
        raise ValueError(f"No rows for sub question '{subq}'. Check the SubQ parsing in Section 4.")

    # Master model diamonds. Out of range positions ('>20', '<0') cannot be plotted.
    m = mm_init[mm_init["SubQ"] == subq].copy()
    m["MM.prob"] = pd.to_numeric(m["MM.prob"], errors="coerce")
    m = m[np.isfinite(m["MM.prob"])]

    # Tipping point labels. The text is kept in its own column so out of range strings survive.
    t = tps[tps["SubQ"] == subq].copy()
    t["lab"] = t["Result"].map(tp_label)
    t["_prio"] = t["DataType"].map(PRIORITY)
    t = t.sort_values("_prio").drop_duplicates(subset=["ERR", "GatewayDesc", "lab", "x_pos"])

    facets = _facet_order(d)
    x_all = 100 * d["Prob"]
    xr = (float(x_all.min()), float(x_all.max()))
    yr = (float(d["Result"].min()), float(d["Result"].max()))
    x_span = (xr[1] - xr[0]) or 1.0
    y_span = (yr[1] - yr[0]) or 1.0
    ypad = 0.10 * y_span

    # Space at the foot of the page: the legend alone, or the legend plus the takeaway block
    bottom = 0.34 if takeaways else 0.09
    fig, axes = plt.subplots(len(facets), 1, figsize=figsize, sharex=True, sharey=True)
    axes = np.atleast_1d(axes)
    fig.subplots_adjust(left=0.13, right=0.86, top=0.94, bottom=bottom, hspace=0.12)

    for ax, (err, gwd) in zip(axes, facets):
        sub = d[(d["ERR"] == err) & (d["GatewayDesc"] == gwd)]
        ax.axhline(0, color=pal["zero"], lw=0.8, zorder=1)      # the benefit / deficit boundary
        ax.grid(True, color=pal["grid"], lw=0.5, zorder=0)
        ax.set_axisbelow(True)

        for dt, style, col in (("Mean", "-", pal["mean"]),
                               ("95%_PI", "--", pal["pi"]),
                               ("95%_PI2", "--", pal["pi"])):
            s = sub[sub["DataType"] == dt].sort_values("Prob")
            ax.plot(100 * s["Prob"], s["Result"], style, color=col, lw=1.1, zorder=3)

        mm_s = m[(m["ERR"] == err) & (m["GatewayDesc"] == gwd)]
        ax.plot(mm_s["MM.prob"], mm_s["MM.survivor"], "D", color=pal["mm"], ms=6, zorder=5)

        # Upper bound above the zero line, mean and lower bound below it, the two bounds pushed
        # sideways in opposite directions by `ldir`. Anything landing on a label already placed is
        # pushed further out, and everything is clamped inside the panel.
        lab = t[(t["ERR"] == err) & (t["GatewayDesc"] == gwd)].copy()
        lab["ny"] = np.where(lab["DataType"] == "95%_PI2", 0.08 * y_span, -0.08 * y_span)
        lab["nx"] = np.where(lab["DataType"] == "95%_PI", -0.05 * x_span * ldir,
                             np.where(lab["DataType"] == "95%_PI2", 0.05 * x_span * ldir, 0.0))
        ylo, yhi = yr[0] - ypad, yr[1] + ypad
        placed = []
        for _, row in lab.sort_values("_prio").iterrows():
            ly = row["ny"] * frc ** 0.5
            while any(abs(ly - q) < 0.11 * y_span for q in placed):
                ly += (0.12 * y_span) * (1 if ly >= 0 else -1)
            placed.append(ly)
            ly = float(np.clip(ly, ylo + 0.07 * (yhi - ylo), yhi - 0.07 * (yhi - ylo)))
            lx = float(np.clip(row["x_pos"] + row["nx"],
                               xr[0] + 0.05 * x_span, xr[1] - 0.05 * x_span))
            anchor_y = float(np.clip(0.0, ylo, yhi))    # keeps '<0%' from being clipped away
            ax.annotate(row["lab"], xy=(row["x_pos"], anchor_y), xytext=(lx, ly), fontsize=8.5,
                        ha="center", va="center", zorder=6, annotation_clip=False,
                        arrowprops=dict(arrowstyle="-", lw=0.5, color=pal["zero"]))

        ax.set_ylim(ylo, yhi)
        ax.tick_params(labelsize=8)

        strip = ax.twinx()                               # the ggplot row strip
        strip.set_yticks([])
        strip.set_ylabel(f"RERR={float(err):.2f}\n{gwd}", rotation=270, fontsize=8,
                         labelpad=32, va="center")
        for sp in strip.spines.values():
            sp.set_visible(False)

    axes[-1].set_xticks(xbreak if xbreak else sorted(set(np.round(100 * d["Prob"], 6))))
    axes[-1].set_xlabel(d["xlabel"].iloc[0], fontsize=9)
    fig.text(0.035, (1 + bottom) / 2 - 0.03, "Difference in survivors",
             rotation=90, va="center", fontsize=9)
    axes[0].set_title(title or d["Strength"].iloc[0], fontsize=9,
                      bbox=dict(facecolor=pal["strip"], edgecolor="none", pad=4))

    handles = [Line2D([], [], color=pal["mean"], ls="-", label="Mean"),
               Line2D([], [], color=pal["pi"], ls="--", label="95% posterior interval"),
               Line2D([], [], color=pal["mm"], marker="D", ls="", label="Master model")]
    fig.legend(handles=handles, loc="lower center", ncol=3, frameon=False, fontsize=8.5,
               bbox_to_anchor=(0.5, bottom - 0.075))

    if takeaways:
        # Step the font down until the wrapped text fits the space left below the legend.
        # Matplotlib will happily draw text off the bottom of the page without complaining.
        head_y = bottom - 0.115
        available = head_y - 0.035
        size, wrapped = 7.6, None
        for size in (7.6, 7.2, 6.8, 6.4, 6.0, 5.6):
            width = int(118 * 7.6 / size)
            wrapped = [textwrap.fill(line, width, subsequent_indent="   ") for line in takeaways]
            if sum(w.count("\n") + 1 for w in wrapped) * 1.5 * (size / 72.0) / figsize[1] <= available:
                break
        fig.text(0.06, bottom - 0.09, "Key takeaways", fontsize=9, weight="bold", va="top")
        fig.text(0.06, head_y, "\n".join(wrapped), fontsize=size, va="top", linespacing=1.5)

    return fig

## Section 7. Saving the figures

**Rationale.** The report needs colour for slides and greyscale for print, and both must come from
the same numbers, so one function builds each figure twice from the same frames rather than anyone
recolouring anything by hand. `.svg` matches the R script's `svg()` device and stays vector for the
report; `.png` is there for pasting into email and slides.

Every figure is closed in a `finally`, which is the equivalent of the R script's `dev.off()`. An
unclosed figure in a notebook is both a memory leak and a duplicate display, and matplotlib warns
once more than twenty are open.

In [ ]:
def save_individual_plots(dat: pd.DataFrame, mm_init: pd.DataFrame, tps: pd.DataFrame,
                          out_dir, product: str, palettes=("color", "gray"),
                          formats=("svg", "png"), dpi: int = None) -> dict:
    """Write every sub question figure to a folder per palette. Returns {palette: [paths]}."""
    out_dir = Path(out_dir)
    dpi = dpi or FIG_DPI
    folders = {"color": out_dir / COLOR_DIR_NAME, "gray": out_dir / GRAY_DIR_NAME}
    written = {p: [] for p in palettes}

    for palette in palettes:
        folder = folders.get(palette, out_dir / palette)
        folder.mkdir(parents=True, exist_ok=True)

        for subq in FIGURE_ORDER:
            if subq not in set(dat["SubQ"]):
                warnings.warn(f"Sub question '{subq}' is not in the export, figure skipped.")
                continue
            tag, xbreak, ldir, frc = FIGSPEC[subq]
            fig = build_figure(dat, mm_init, tps, subq, palette=palette,
                               xbreak=xbreak, ldir=ldir, frc=frc)
            stem = f"{safe_name(product)}_{tag}_Mean95PIa"
            try:
                for ext in formats:
                    path = folder / f"{stem}.{ext}"
                    fig.savefig(path, dpi=dpi, format=ext)
                    written[palette].append(path)
            finally:
                plt.close(fig)          # the equivalent of dev.off(), always runs

    for palette, paths in written.items():
        print(f"  {palette:<6} {len(paths):>2} file(s) -> {folders.get(palette, out_dir / palette)}")
    return written

## Section 8. Key takeaways

**Rationale.** The sentences on the PDF page are generated from the calculated tipping points rather
than typed, so they cannot drift away from the figure beside them when the data is refreshed.

Each panel gets one line: the mean tipping point with its interval, or a plain statement that the
curve does not cross zero inside the swept range. The direction wording comes from the sign of the
fitted slope, so a falling curve reads as a benefit lost above the tipping point and a rising one as
a benefit gained. The master model position is on the same line, since that is what the diamond is.

In [ ]:
def _direction_words(slope: float) -> tuple[str, str]:
    """Wording for a falling or rising curve."""
    if not np.isfinite(slope):
        return "changes", "beyond"
    return ("falls", "above") if slope < 0 else ("rises", "below")


def build_key_takeaways(subq: str, dat: pd.DataFrame, tps: pd.DataFrame,
                        mm_init: pd.DataFrame) -> list[str]:
    """One bullet per panel plus a summary line, describing what the figure shows."""
    d = dat[dat["SubQ"] == subq]
    t = tps[tps["SubQ"] == subq]
    m = mm_init[mm_init["SubQ"] == subq]
    if d.empty or t.empty:
        return [f"No data for sub question {subq}."]

    probs = sorted(set(np.round(100 * d["Prob"], 6)))
    lines = [f"{TRANSITIONS.get(subq, subq)} ({subq}) swept from {num_str(min(probs))}% to "
             f"{num_str(max(probs))}% over {len(probs)} probabilities, "
             f"{MORTALITY_MODEL_PLOT.lower()} mortality model."]

    crossing = 0
    for err, gwd in _facet_order(d):
        gate = "10%" if "10%" in gwd else "25%"
        sel = t[(t["ERR"] == err) & (t["GatewayDesc"] == gwd)]
        mean_row = sel[sel["DataType"] == "Mean"]
        if mean_row.empty:
            continue
        mean_row = mean_row.iloc[0]
        lo = sel.loc[sel["DataType"] == "95%_PI", "Result"]
        hi = sel.loc[sel["DataType"] == "95%_PI2", "Result"]
        verb, side = _direction_words(mean_row["Slope"])

        mm_sel = m[(m["ERR"] == err) & (m["GatewayDesc"] == gwd)]
        mm_txt = ""
        if not mm_sel.empty:
            mm_val = mm_sel.iloc[0]["MM.prob"]
            shown = mm_val if isinstance(mm_val, str) else tp_label(mm_val)
            mm_txt = (f" Master model result ({mm_sel.iloc[0]['MM.survivor']:,.0f} survivors) is "
                      f"reached at {shown}.")

        if isinstance(mean_row["Result"], str):
            lines.append(
                f"RERR {float(err):.2f}, gateway {gate}: the mean curve does not cross zero inside "
                f"the swept range; the difference {verb} with the swept probability, so the "
                f"tipping point is {mean_row['Result']}.{mm_txt}")
        else:
            crossing += 1
            band = ""
            if len(lo) and len(hi):
                band = f" (95% PI {tp_label(lo.iloc[0])} to {tp_label(hi.iloc[0])})"
            lines.append(
                f"RERR {float(err):.2f}, gateway {gate}: mean tipping point "
                f"{tp_label(mean_row['Result'])}{band}. The survivor difference {verb} as the "
                f"probability rises, so the modelled benefit is lost {side} that point.{mm_txt}")

    lines.append(f"{crossing} of {len(_facet_order(d))} panels cross zero inside the swept range; "
                 "a tipping point shown as '<' or '>' lies outside it and is pinned to that end of "
                 "the axis on the figure.")
    return lines

## Section 9. The PDF report

**Rationale.** One file that can be sent to someone who does not have the folder: page 1 records who
ran it, when, on which version and from which inputs, and every figure after that carries its own
takeaways. Provenance next to the numbers is what makes a figure re-checkable months later.

The figures are rebuilt into the PDF rather than embedded from the saved files, so they stay vector
and stay searchable. The text page measures its own content and steps the font down until it fits,
because matplotlib draws text off the bottom of a page without complaining.

In [ ]:
def resolve_author(name=None, initials=None, path=None) -> str:
    """Explicit name, else the initials from Model Group, else the account name from the path."""
    if name:
        return str(name)
    if initials:
        return str(initials)
    parts = [p for p in re.split(r"[\\/]+", str(path or Path.cwd())) if p]
    for i, part in enumerate(parts[:-1]):
        if part.casefold() in ("users", "home") and parts[i + 1].casefold() not in (
                "downloads", "documents", "desktop", "public"):
            candidate = re.sub(r"(?<=[a-z0-9])(?=[A-Z])", " ", parts[i + 1])
            return " ".join(w[:1].upper() + w[1:] for w in re.split(r"[._\-\s]+", candidate) if w)
    try:
        return getpass.getuser()
    except Exception:
        return "Unknown"


def _pdf_text_page(pdf, title: str, blocks, figsize=None, footer: str = None) -> None:
    """Write one text only page: a title, then labelled blocks of lines, auto fitted."""
    figsize = figsize or (FIG_WIDTH, FIG_HEIGHT)
    top, floor = 0.87, 0.10 if footer else 0.05

    def layout(size):
        width = max(int(92 * 9.0 / size), 40)
        line_h = 1.45 * (size / 72.0) / figsize[1]
        head_h = 1.45 * ((size + 1.5) / 72.0) / figsize[1]
        out, total = [], 0.0
        for heading, lines in blocks:
            wrapped = [textwrap.fill(ln, width, subsequent_indent="    ") for ln in lines]
            out.append((heading, wrapped))
            total += head_h + 0.012
            total += sum(line_h * (w.count("\n") + 1) + 0.006 for w in wrapped)
            total += 0.014
        return out, total

    size = 9.0
    laid, height = layout(size)
    while height > (top - floor) and size > 5.5:
        size -= 0.5
        laid, height = layout(size)

    fig = plt.figure(figsize=figsize)
    try:
        fig.text(0.08, 0.94, title, fontsize=16, weight="bold", va="top")
        fig.add_artist(Line2D([0.08, 0.92], [0.915, 0.915], color="#E26B0A", lw=2))
        line_h = 1.45 * (size / 72.0) / figsize[1]
        head_h = 1.45 * ((size + 1.5) / 72.0) / figsize[1]
        y = top
        for heading, wrapped in laid:
            fig.text(0.08, y, heading, fontsize=size + 1.5, weight="bold", va="top")
            y -= head_h + 0.012
            for w in wrapped:
                fig.text(0.10, y, w, fontsize=size, va="top", linespacing=1.45)
                y -= line_h * (w.count("\n") + 1) + 0.006
            y -= 0.014
        if footer:
            fig.text(0.08, 0.045, textwrap.fill(footer, int(110 * 9.0 / size)),
                     fontsize=size - 1.5, color="#666666", va="bottom")
        pdf.savefig(fig)
    finally:
        plt.close(fig)


def build_pdf_report(dat, mm_init, tps, out_dir, context, pdf_name: str = None) -> Path:
    """Write the multipage PDF: metadata page, then one page per colour figure."""
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    path = out_dir / (pdf_name or PDF_NAME)

    subqs = [s for s in FIGURE_ORDER if s in set(dat["SubQ"])]
    outputs = [f"{COLOR_DIR_NAME}/ and {GRAY_DIR_NAME}/ : {len(subqs)} figures each, .svg and .png",
               f"{safe_name(context['product'])}_TippingPoints.csv : QC table of every tipping point",
               f"{path.name} : this report",
               f"{EXPORT_DIR_STEM}_v{PIPELINE_VERSION}_<date>/ : a frozen copy of all of the above"]

    with PdfPages(path) as pdf:
        _pdf_text_page(
            pdf,
            f"Tipping Point 1D Plotting, {context['product']}",
            [("Run details", [
                f"Author: {context['author']}    Analyst initials: {context['initials']} "
                f"(from Model Group)",
                f"Product: {context['product']}",
                f"Date: {context['run_dt'].strftime('%d %b %Y')}    "
                f"Time: {context['run_dt'].strftime('%H:%M:%S')}    "
                f"Timestamp: {context['stamp']}",
                f"Version: v{PIPELINE_VERSION} ({VERSION_TYPE})",
                f"Environment: Python {platform.python_version()}, pandas {pd.__version__}, "
                f"numpy {np.__version__}, matplotlib {matplotlib.__version__}",
                f"Filters: node {NODE_KEEP}, age range {AGE_RANGE_KEEP}, "
                f"RERR {', '.join(f'{e:.2f}' for e in ERR_KEEP)}, "
                f"{MORTALITY_MODEL_PLOT} mortality model",
             ]),
             ("Objective", [context["objective"]]),
             ("Input files", [f"{i}. {n}   [sheet: {SHEET}]"
                              for i, n in enumerate(context["input_files"], start=1)]),
             ("Output files", [f"{i}. {n}" for i, n in enumerate(outputs, start=1)]),
             ("Contents", [f"Page {i + 2}: Figure {FIGURE_NUMBERS.get(s, '')}, "
                           f"{TRANSITIONS.get(s, s)} ({s})" for i, s in enumerate(subqs)]),
             ("How to read a figure", [
                 "Each panel is one RERR and gateway combination. The solid line is the mean "
                 "difference in survivors, the dashed lines are the 95% posterior interval bounds.",
                 "Where a curve crosses the zero line is its tipping point, printed beside the "
                 "crossing. A label shown as '<' or '>' means the curve does not cross zero "
                 "anywhere inside the swept range.",
                 "The diamond marks the swept probability at which the mean curve reaches the "
                 "master model survivor value.",
             ])],
            footer=f"Input folder: {context['data_dir']}    Output folder: {out_dir}")

        for subq in subqs:
            tag, xbreak, ldir, frc = FIGSPEC[subq]
            fig = build_figure(
                dat, mm_init, tps, subq, palette="color", xbreak=xbreak, ldir=ldir, frc=frc,
                takeaways=build_key_takeaways(subq, dat, tps, mm_init),
                title=f"Figure {FIGURE_NUMBERS.get(subq, '')}. {TRANSITIONS.get(subq, subq)} "
                      f"({subq}), {context['product']}")
            try:
                pdf.savefig(fig)
            finally:
                plt.close(fig)

        meta = pdf.infodict()
        meta["Title"] = f"Tipping Point 1D Plotting, {context['product']}"
        meta["Author"] = context["author"]
        meta["Subject"] = context["objective"]
        meta["Keywords"] = "DPM, tipping point, " + ", ".join(TRANSITIONS[s] for s in subqs)
        meta["CreationDate"] = context["run_dt"]

    print(f"  PDF    {1 + len(subqs)} page(s) -> {path}")
    return path

## Section 10. QC table

**Rationale.** The figures print tipping points but cannot be copied from. This is the same numbers
in one flat file, one row per transition, gateway, RERR and statistic, which is what to work from
when transcribing into the report text or checking against the DPM Tipping Point Analysis page.

In [ ]:
def build_qc_table(tps: pd.DataFrame) -> pd.DataFrame:
    """Flat table of every tipping point, one row per transition, gateway, RERR and statistic."""
    qc = tps.copy()
    qc["Transition"] = qc["SubQ"].map(TRANSITIONS)
    qc["Statistic"] = qc["DataType"].map(STAT_LABELS)
    qc = qc.rename(columns={"Result": "TippingPoint"})
    qc = qc[["Product", "Transition", "SubQ", "Gateway", "Mortality_Model", "ERR",
             "Statistic", "TippingPoint"]]
    return qc.sort_values(["SubQ", "Gateway", "ERR", "Statistic"]).reset_index(drop=True)

## Section 11. Orchestrator and run

**Rationale.** One call, so there is no order to remember and no chance of running the figures
against one set of frames and the QC table against another. It returns every intermediate frame, so
any step can be inspected afterwards without rerunning.

Close both workbooks in Excel before running, an open file can be locked against reading.

In [ ]:
def run_pipeline(data_dir=None, out_dir=None, author=None, pdf_name=None) -> dict:
    """Resolve the inputs, calculate, plot, and write every output. Returns the run."""
    # Section 3, Requirements 1 and 2
    resolved = resolve_inputs(data_dir or DATA_DIR, out_dir or OUT_DIR)
    product, out_dir = resolved["product"], resolved["out_dir"]

    # Section 4, prepare both exports
    dat = prepare_sweeps(resolved["frames"]["sweeps"], product)
    mmdat = prepare_master(resolved["frames"]["master"], product)

    # Section 5, tipping points
    tps = calc_tps(dat)
    mm_init = calc_tps_mm(dat, mmdat)
    print(f"Tipping points calculated for {tps['Key'].nunique()} sweeps.")

    # Sections 7 and 10, figures and QC table
    out_dir.mkdir(parents=True, exist_ok=True)
    figures = save_individual_plots(dat, mm_init, tps, out_dir, product)

    qc = build_qc_table(tps)
    qc_path = out_dir / f"{safe_name(product)}_TippingPoints.csv"
    qc.to_csv(qc_path, index=False)
    print(f"  QC     {len(qc)} row(s) -> {qc_path}")

    # Section 9, the report
    run_dt = datetime.now()
    context = {"author": resolve_author(author or AUTHOR_NAME, resolved["initials"],
                                        resolved["data_dir"]),
               "initials": resolved["initials"],
               "product": product,
               "run_dt": run_dt,
               "stamp": run_dt.strftime("%d%b%Y_%H%M%S"),
               "objective": ANALYSIS_OBJECTIVE,
               "input_files": [resolved["paths"]["sweeps"].name, resolved["paths"]["master"].name],
               "data_dir": resolved["data_dir"]}
    pdf_path = build_pdf_report(dat, mm_init, tps, out_dir, context, pdf_name=pdf_name)

    return {"resolved": resolved, "dat": dat, "mmdat": mmdat, "tps": tps, "mm_init": mm_init,
            "qc": qc, "figures": figures, "qc_file": qc_path, "pdf_file": pdf_path,
            "product": product, "out_dir": out_dir, "run_dt": run_dt, "context": context}

In [ ]:
result = run_pipeline()

# Mean tipping points only, one line per transition, gateway and RERR. The full table, including
# both interval bounds, is in the CSV.
print("\nMean tipping points (%):")
print(result["qc"].query("Statistic == 'Mean'")[
          ["Transition", "Gateway", "ERR", "TippingPoint"]].to_string(index=False))

In [ ]:
# The same text the PDF pages carry, one block per figure
for subq in FIGURE_ORDER:
    print(f"\n=== Figure {FIGURE_NUMBERS[subq]}, {TRANSITIONS[subq]} ({subq}) ===")
    for line in build_key_takeaways(subq, result["dat"], result["tps"], result["mm_init"]):
        print(textwrap.fill(line, 110, initial_indent="  ", subsequent_indent="     "))

## Section 12. Export bundle

**Rationale.** The working outputs are overwritten on every run, which is right for iterating and
wrong for anything that has been circulated. This copies the finished run into a folder stamped with
the version and the date, so each version is preserved and can be pointed at unambiguously.

```
<OUT_DIR>/Exports_v3.0_04Sep2026/
    color/  gray/       the eight figures
    <Product>_TippingPoints.csv
    Tipping Point_1D_Plotting_&_keyTakeways.pdf
    MANIFEST.txt        what was copied, when, by whom, from which inputs
```

Bumping `PIPELINE_VERSION` or running on another day creates a new folder; a rerun on the same day at
the same version refreshes that one in place.

In [ ]:
def export_outputs(result: dict, version: str = None, when=None, stem: str = None,
                   extra_paths=()) -> Path:
    """Copy every output of a run into Exports_v<version>_<DDMonYYYY> and write a manifest."""
    out_dir = Path(result["out_dir"])
    version = version or PIPELINE_VERSION
    when = when or result.get("run_dt") or datetime.now()
    dest = out_dir / f"{stem or EXPORT_DIR_STEM}_v{version}_{when.strftime('%d%b%Y')}"
    dest.mkdir(parents=True, exist_ok=True)

    copied = []
    for folder in (out_dir / COLOR_DIR_NAME, out_dir / GRAY_DIR_NAME):
        if folder.exists() and folder.resolve() != dest.resolve():
            target = dest / folder.name
            shutil.copytree(folder, target, dirs_exist_ok=True)      # refresh rather than fail
            copied += sorted(p for p in target.rglob("*") if p.is_file())

    for path in [result["qc_file"], result["pdf_file"], *extra_paths]:
        path = Path(path)
        if path.exists():
            copied.append(Path(shutil.copy2(path, dest / path.name)))

    lines = [f"Tipping Point 1D Plotting, {result['product']}",
             f"Pipeline version : v{version} ({VERSION_TYPE})",
             f"Exported         : {when.strftime('%d %b %Y %H:%M:%S')}",
             f"Author           : {result['context']['author']}   "
             f"(initials {result['context']['initials']})",
             f"Input files      : {', '.join(result['context']['input_files'])}",
             f"Input folder     : {result['context']['data_dir']}",
             f"Output folder    : {out_dir}",
             "", f"Files ({len(copied)}):"]
    lines += [f"  {p.relative_to(dest)}" for p in copied]
    (dest / "MANIFEST.txt").write_text("\n".join(lines), encoding="utf-8")

    print(f"Exported {len(copied)} file(s) -> {dest}")
    for p in copied:
        print(f"  {p.relative_to(dest)}")
    return dest


export_dir = export_outputs(result)

## Section 13. Validation

**Rationale.** A pipeline that runs is not the same as a pipeline that is right. These checks assert
the things that would otherwise fail quietly: a filter that silently dropped every row of a sub
question, a tipping point that does not sit between the two probabilities it was interpolated from,
a figure file that was written empty because it was closed before saving.

They run on the real inputs, so this section is also the test run. If every assertion passes, the
outputs are consistent with the data they came from.

In [ ]:
qc, dat, tps = result["qc"], result["dat"], result["tps"]
n_facets = len(_facet_order(dat))

# 1. Coverage. Every sub question, gateway, RERR and statistic must be present exactly once.
expected_rows = len(set(dat["SubQ"])) * n_facets * 3
assert len(qc) == expected_rows, f"QC table has {len(qc)} rows, expected {expected_rows}"
assert set(dat["SubQ"]) == set(FIGURE_ORDER), f"Missing sub questions: {set(FIGURE_ORDER) - set(dat['SubQ'])}"
assert not qc.duplicated(subset=["SubQ", "Gateway", "ERR", "Statistic"]).any()

# 2. Every numeric tipping point must lie inside the sweep it came from, and every out of range
#    label must be pinned to the correct end of that sweep.
for _, row in tps.iterrows():
    if isinstance(row["Result"], str):
        end = row["SweepMax"] if row["Result"].startswith(">") else row["SweepMin"]
        assert np.isclose(row["x_pos"], end), f"{row['Key']} {row['DataType']} pinned to the wrong end"
    else:
        assert row["SweepMin"] <= row["Result"] <= row["SweepMax"], \
            f"{row['Key']} {row['DataType']} tipping point {row['Result']} is outside its sweep"

# 3. The interpolation itself: the mean curve must change sign across the crossing.
for key, g in dat[dat["DataType"] == "Mean"].groupby("Key"):
    row = tps[(tps["Key"] == key) & (tps["DataType"] == "Mean")].iloc[0]
    if isinstance(row["Result"], str):
        continue
    g = g.sort_values("Probability")
    below = g.loc[g["Probability"] <= row["Result"], "Result"]
    above = g.loc[g["Probability"] >= row["Result"], "Result"]
    assert len(below) and len(above), f"{key} has no points either side of its tipping point"
    assert below.iloc[-1] * above.iloc[0] <= 0, f"{key} does not change sign across its crossing"

# 4. Every master model diamond must join to exactly one survivor value.
assert len(result["mm_init"]) == tps["Key"].nunique(), "A sweep is missing its master model marker"
assert result["mm_init"]["MM.survivor"].notna().all()

# 5. Every promised file exists and is not empty.
for palette, paths in result["figures"].items():
    assert len(paths) == len(FIGURE_ORDER) * 2, f"{palette}: expected 8 files, got {len(paths)}"
    assert all(p.exists() and p.stat().st_size > 0 for p in paths), f"{palette}: an empty figure"
for path in (result["qc_file"], result["pdf_file"], export_dir / "MANIFEST.txt"):
    assert Path(path).exists() and Path(path).stat().st_size > 0, f"Missing or empty: {path}"

print(f"All checks passed: {len(qc)} tipping points across {n_facets} panels and "
      f"{len(set(dat['SubQ']))} transitions, "
      f"{sum(len(v) for v in result['figures'].values())} figure files, "
      f"1 QC table, 1 PDF, 1 export bundle.")
display(qc.head(12))

## Section 14. Troubleshooting

| Message | Cause and fix |
|---|---|
| `Requirement 1 not met: expected exactly one 'DetailedModelResults_<Product>_TippingPoint.xlsx'` | The file is missing, misnamed, or there are two candidates in the folder. Rename to the convention and keep one of each role. |
| `Requirement 1 not met: the two file names carry different products` | For example `..._Product_X_TippingPoint.xlsx` beside `..._ProductX_MasterModels.xlsx`. Spell the product identically in both. |
| `Requirement 2 not met: ... its Model Group is ...` | The `Model Group` column does not read `<Product>_<initials>`, or names a different product than the file name. Fix whichever is wrong. |
| `Model Group is '...', which does not meet Requirement 2` | Only one token, so there are no initials to split off. It must read for example `Product_X_YO`. |
| `The two workbooks do not hold what their names claim` | The two exports are saved under each other's names. A TippingPoint file's model names end in the swept probability, a MasterModels file's in a cohort tag such as `B01`. |
| `contains N model groups` | This is the one product pipeline. Split the export. |
| `Could not locate the '95% PI' column pair` | The export is missing its title row, so the header landed on the wrong row. Row 1 should read `Detailed Model Results`, row 2 the column names. |
| `is missing the required column '...'` | An export was edited and a column was renamed or removed. |
| `No rows left after filtering the ... export` | One of `NODE_KEEP`, `AGE_RANGE_KEEP`, `ERR_KEEP` or `MORTALITY_MODEL_PLOT` does not match the workbook. Print `read_dpm(path)["Node"].unique()` and the same for `Age_Range` and `ERR` to see what is there. |
| `rows have no sub question tag` | A model name is missing its `5a`, `6a`, `14b` or `15a` tag, usually a renamed batch. |
| `rows have an unparseable swept probability` | A model name does not end in a number. Section 3 catches the usual cause, a master model file read as a sweep file, so this points at the naming inside the export. |
| `rows ... have neither 'G10' nor 'G25' in the model name` | A gateway is missing from the model names, so its panel cannot be identified. |
| `The tipping point and master model exports did not join on any row` | The two workbooks do not cover the same RERRs, gateways or mortality model. Both pass through the same Section 4 filters, so compare them after filtering. |
| `Key ... matched N master model means` | The master model export has a duplicate run for one gateway and RERR. Remove it. |
| A tipping point reads `<0%` or `>20%` | The curve does not cross zero anywhere inside the swept range. That is a result, not an error; the label is pinned to the end of the range it falls beyond. |
| Labels overlap or sit awkwardly | Adjust `ldir` (sideways push) and `frc` (separation force) for that sub question in `FIGSPEC`, Section 1. |
| A figure or the PDF is blank | Almost always a figure closed before it was saved. Both writers save inside a `try` and close in the `finally`, which is the equivalent of the R script's explicit `print()` before `dev.off()`. |
| `More than 20 figures have been opened` | A run was interrupted between building and closing a figure. Run `plt.close("all")` and rerun. |
| `Permission denied` when writing | An output file is open in Excel. Close it and rerun. |